In [78]:
import main
import pandas as pd
import numpy as np

SapModel = main.main() # always select 0 when using with notebook

import ETABSv1 as etabs


Attached file is M9856-M08-ULS-R00-t06.EDB


In [3]:
from csiapi import csiutils
csiutils.clear_selection(SapModel) # test

0

In [79]:
def all_column_design_forces(sapmodel): #this will pull all the data from db, quite faster than reading line by line

    result = sapmodel.DatabaseTables.GetTableForDisplayArray(
        "Design Forces - Columns",
        [str()],
        str(),
        int(),
        [str()],
        int(),
        [str()]
    )

    ret = result[0]

    if ret != 0:
        raise RuntimeError("Could not retrieve column design forces table")

    # table_version = result[1]
    field_keys = list(result[1])
    # group_name = result[3]
    # num_records = result[4]
    table_data = list(result[5])

    if len(field_keys)>1:
        n_fields = len(field_keys)
    else:
        print("No field key identified, user defined headers will be used")
        field_keys = ["Story","Column","Unique_Name","Combo","Station","P","V2","V3","T","M2","M3"]
        n_fields = len(field_keys)

    # reshape 1D list to 2D
    data_array = np.array(table_data).reshape(-1, n_fields)

    df = pd.DataFrame(data_array, columns=field_keys)
    if "Frame" in df.columns:
        df.rename(columns={"Frame": "Unique_Name"}, inplace=True)

    numeric_cols = ["P", "V2", "V3", "T", "M2", "M3"]

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col])

    return df

In [134]:
if SapModel.DesignConcrete.GetResultsAvailable(): #if results available we will no design again
    etabs.cDesignConcrete(SapModel.DesignConcrete)
    df = all_column_design_forces(SapModel)
else:
    from csiapi import ops
    design_concrete = ops.DesignConcrete(SapModel)
    df = design_concrete.all_column_design_forces()

No field key identified, user defined headers will be used


In [ ]:
# function to get ratio of moments
group_cols = ["Story", "Column", "Unique_Name", "Combo"]

def get_ratio(g, moment_col):
    first = g.loc[g["Station"].idxmin(), moment_col]
    last  = g.loc[g["Station"].idxmax(), moment_col]

    return first/last if abs(first) < abs(last) else last/first

ratio_m2 = (
    df.groupby(group_cols)[["Station","M2"]]
      .apply(lambda g: get_ratio(g, "M2"))
      .rename("M2_ratio")
)

ratio_m3 = (
    df.groupby(group_cols)[["Station","M3"]]
      .apply(lambda g: get_ratio(g, "M3"))
      .rename("M3_ratio")
)

df = df.merge(ratio_m2, on=group_cols)
df = df.merge(ratio_m3, on=group_cols)

In [188]:
#Finding max compressin
compression = df.groupby("Unique_Name")["P"].min()
compression.sort_values().head(10)

Unique_Name
291   -7893.8612
272   -6571.3593
186   -5451.2575
281   -5441.2699
168   -5231.3448
278   -5215.6939
267   -5127.2551
164   -5021.8425
279   -4819.7696
268   -4586.3100
Name: P, dtype: float64

In [243]:
#Finding force values in a member for it maximum compression
idx = df.groupby("Unique_Name")["P"].idxmin()
col_P = df.loc[idx] # Dataframe with which load combo governing each column (in terms of compression)
col_P.sort_values("P", ascending=True).head(20)
col_P[col_P.Unique_Name == "271"]

,Story,Column,Unique_Name,Combo,Station,P,V2,V3,T,M2,M3,M2_ratio,M3_ratio
1783780,Ground,C186,271,OS-UQ01_4_1 - 1.2DL+LL-2.5EQy +0.75EQx (STAGED)-5,0,-3453.8948,-53.833,-0.5016,-2.6371,0.8347,-196.9429,0.35844,-0.19633


In [192]:
#study
col_P["Combo"].value_counts().head(10) # which combo governs design

Combo
OS-R-UQ01_3_1 - 1.2DL+LL+2.5SpecY +0.75SpecX (STAGED)-5               10
OS-R-UQ01_1_1 - 1.2DL+LL+2.5SpecX+0.75SpecY-5                          8
OS-R-UQ01_3_3 - 1.2DL+LL+1.6H+2.5SpecY +0.75SpecX (STAGED)-5           8
OS-R-UQ01_1_4 - 1.2DL+LL+1.2FL+1.6H+2.5SpecX +0.75SpecY (STAGED)-5     7
OS-R-UQ01_1_3 - 1.2DL+LL+1.6H+2.5SpecX +0.75SpecY (STAGED)-5           7
UTH-08 - 1.2DL+1.6LL+0.5LLROOF-1.2T (STAGED)-1                         7
OS-R-UQ01_1_2 - 1.2DL+LL+1.2FL+2.5SpecX +0.75SpecY (STAGED)-5          7
OS-R-UQ01_1_1 - 1.2DL+LL+2.5SpecX +0.75SpecY (STAGED)-5                5
OS-R-UQ01_3_1 - 1.2DL+LL+2.5SpecY+0.75SpecX-5                          5
UTH-01 - 1.2DL+1.6LL+0.5LLROOF+1.6H+1.2FL+1.2T (STAGED)-1              5
Name: count, dtype: int64

In [18]:
#study
story_load = col_P.groupby("Story")["P"].sum()
story_load.sort_index()

Story
1st       -65966.0033
2nd       -30889.9442
Ground   -109149.6560
Roof       -5283.9293
Name: P, dtype: float64

In [19]:
#Column with high moments
df["Mmax"] = df[["M2", "M3"]].abs().max(axis=1)
df["M_direction"] = df[["M2","M3"]].abs().idxmax(axis=1)
col_moment = df.loc[df.groupby("Unique_Name")["Mmax"].idxmax()]
col_moment.sort_values("Mmax", ascending=False).head(20)


,Story,Column,Unique_Name,Combo,Station,P,V2,V3,T,M2,M3,Mmax,M_direction
1101640,1st,C107,167,OS-R-UQ01_3_1 - 1.2DL+LL+2.5SpecY+0.75SpecX-1,0,-1112.1099,1995.4941,301.1830,70.7354,682.5900,7734.4927,7734.4927,M3
1072632,1st,C106,166,OS-R-UQ01_3_4 - 1.2DL+LL+1.2FL+1.6H+2.5SpecY+0...,0,-3317.0942,2476.8759,47.8430,70.9613,131.7477,7268.9759,7268.9759,M3
1019256,1st,C104,164,OS-R-UQ01_1_4 - 1.2DL+LL+1.2FL+1.6H+2.5SpecX+0...,0,-3732.0821,1381.4406,37.0997,17.3841,94.1404,4073.2431,4073.2431,M3
1282136,1st,C191,223,OS-R-UQ01_1_4 - 1.2DL+LL+1.2FL+1.6H+2.5SpecX+0...,0,-1032.3412,1154.3465,210.3764,39.0207,604.8562,3338.7707,3338.7707,M3
941199,1st,C90,1,OS-R-UQ01_3_4 - 1.2DL+LL+1.2FL+1.6H+2.5SpecY+0...,0,-2487.1386,-533.8554,527.7166,71.1273,-2221.3983,196.8953,2221.3983,M2
308408,2nd,C114,178,OS-R-UQ01_1_3 - 1.2DL+LL+1.6H+2.5SpecX+0.75Spe...,0,-758.0095,603.5029,326.0424,25.1300,752.7717,2207.7221,2207.7221,M3
338088,2nd,C117,180,OS-R-UQ01_1_3 - 1.2DL+LL+1.6H+2.5SpecX+0.75Spe...,0,-721.4445,560.8600,174.7821,25.3183,392.3284,2133.8307,2133.8307,M3
970648,1st,C91,2,OS-R-UQ01_3_3 - 1.2DL+LL+1.6H+2.5SpecY+0.75Spe...,0,-1932.3106,-156.3069,917.9379,71.5815,2058.0703,-226.4985,2058.0703,M2
822248,2nd,C10,187,OS-R-UQ01_3_3 - 1.2DL+LL+1.6H+2.5SpecY+0.75Spe...,0,-1286.8735,834.0311,66.9226,19.4475,101.3865,1884.6025,1884.6025,M3
1269992,1st,C190,222,OS-R-UQ01_3_2 - 1.2DL+LL+1.2FL+2.5SpecY+0.75Sp...,0,-886.2922,245.0743,653.6738,53.5472,1871.4760,849.1021,1871.4760,M2
